# Midjourney prompts: exploration and validity check

This notebook is for **looking at the data**, not for producing the paper's
numbers. Every number, table and figure in the paper comes from
`pipeline.py`. This notebook:

1. loads the data with the pipeline's own functions, so it uses exactly the
   same rules (`features.py`);
2. shows example rows behind each step of the cleaning and each research
   question;
3. sets up the **100-prompt hand check** for Discussion › Validity, and scores
   it once both of us have labelled it.

Run it top to bottom (*Restart & Run All*) before submitting, so the saved
outputs match the code. Run `fetch_data.py` and `pipeline.py` first.

In [1]:
import json
import sys
from pathlib import Path

# Works whether the notebook is started from assignment3/ or analysis/.
HERE = Path.cwd() if (Path.cwd() / "pipeline.py").exists() else Path.cwd() / "analysis"
sys.path[:0] = [str(HERE), str(HERE / ".deps")]   # features, pipeline, pyarrow

import numpy as np
import pandas as pd
import features as F
import pipeline as P

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)
SEED = P.SEED
print("data:", HERE / "data" / "raw")

data: /home/elta/repos/applied-research/assignment3/analysis/data/raw


## 1. Load and clean (same code as the pipeline)

`P.load()` checks each file's SHA-256 against `MANIFEST.json`, `P.audit()`
classifies every row, and `P.corpus()` keeps real prompts and removes
duplicates. The last line checks that we got the same corpus the paper
reports.

In [2]:
raw = P.audit(P.load())
c = P.corpus(raw)

reported = json.loads((HERE / "results" / "results.json").read_text())
assert len(c) == reported["flow"]["corpus"], "corpus differs from results.json; re-run pipeline.py"
print(f"{len(raw):,} raw rows -> {len(c):,} corpus prompts (matches results.json)")

246,381 raw rows -> 57,670 corpus prompts (matches results.json)


## 2. What the excluded rows look like

Eight random examples of each row type. Check that the rule in
`features.fragment_type` is doing what the paper says it does: anything that
looks like a real prompt in a non-`prompt` group is a false exclusion.

In [3]:
for kind, group in raw.groupby("fragment"):
    print(f"--- {kind}  ({len(group):,} rows)")
    for t in group["text"].sample(min(8, len(group)), random_state=SEED):
        print("   ", repr(t[:140]))

--- aspect_ratio_only  (34,571 rows)
    '16:9'
    '2:3'
    '24:18'
    '2:3'
    '16:9'
    '9:16'
    '3:4'
    '2:1'
--- empty  (1,597 rows)
    ''
    ''
    ''
    ''
    ''
    ''
    ''
    ''
--- number_only  (13,922 rows)
    '350'
    '3'
    '1200'
    '95'
    '1920'
    '1920'
    '85'
    '.9'
--- prompt  (120,357 rows)
    'ornate, intricate, young, beautiful, woman, Anna Karina, fractal patterns, intricate lace, ethereal, phantasmagoric, flowers on face and hai'
    'building a meaningful world --ar 16:9'
    'contemporary graphic design of a flat fine art oil painting of a perfectly composed traditional town Machias, Maine in the style of Kristin '
    'twoor three pretty women flirting in an imaginary city, in the style of edward hopper nighthawks, bladerunner'
    'historical color photo interior Kowloon Walled City mah jongg game --wallpaper'
    'Optimus Prime robot from Transformers, unreal engine --uplight'
    'chinese art Tesla coil skyscraper, by Alphonse Mu

Short fragments sit right at the cut-off (`MIN_CONTENT_WORDS = 3`), so
they deserve a closer look: are two-word rows like *"Art Nouveau"* lost
prompts, or pieces of a longer message?

In [4]:
short = raw[raw.fragment == "short_fragment"]
short["text"].str.strip().value_counts().head(20)

text
render        1851
style         1068
dof            961
field          747
realistic      689
8K             689
blur           672
8k             648
particles      492
art            463
clay           453
nouveau        444
4k             437
detailed       412
light          406
artstation     391
text           388
detail         386
lighting       348
photograph     329
Name: count, dtype: int64

## 3. Duplicates and split leakage

The most repeated rows, and some test-split prompts that also appear in
train.

In [5]:
raw["norm"].value_counts().head(15).rename("copies").to_frame()

,copies
norm,
16:9,12470
--uplight,9969
9:16,4662
2:3,2479
1920,2054
render,1851
,1597
3:4,1424
1,1391


In [6]:
train_norm = set(raw.loc[raw.split == "train", "norm"])
test_prompts = raw[(raw.split == "test") & (raw.fragment == "prompt")]
leaked = test_prompts[test_prompts["norm"].isin(train_norm)]
print(f"{len(leaked):,} of {len(test_prompts):,} test prompts are also in train")
leaked["text"].sample(8, random_state=SEED).to_frame()

4,167 of 6,090 test prompts are also in train


,text
234770,liquid wooden chainmail ferrofluid:: ultraviolet mage in armour full shot by Annie Leibovitz:: wooden glowing eyes outline:: beautiful neon green braided ha...
243263,"matte painting: :3 clock: :4, medallions: :2, birds, squirrels, snake, renaissance style, hyper realistic animal concept art, detailed and intricate, majest..."
235985,Matrix beautiful boy
241112,"cute furry monster in Pixar with star, colorful, volumetric lighting, 4k, photorealistic"
241943,ridley scott interior medieval speakeasy club --ar 16:8
245567,"silk crystal intricate hyper realistic misty river, alone and frightened, in an empty courtroom, overcast, hazy, rays of light, cinematic, something no one ..."
243221,"a conch shell carved out of fibonacci fractal resonance, golden ratio, realism, hd --no gold color"
234679,"a cube covered in grass and poppies, floating in the middle of a blue sky --ar 2:3"


## 4. Corpus prompts with the variables extracted

A random sample showing what the rules pulled out of each prompt. Read a few
and check the parameters, artists and boosters by eye.

In [7]:
cols = ["text", "n_words", "params", "aspect", "artists_strict", "artists_lenient", "boosters"]
c[cols].sample(10, random_state=SEED)

,text,n_words,params,aspect,artists_strict,artists_lenient,boosters
220885,"ornate wooden spiral staircase, mobius strip, magic realism, intricate detail, elegant design --aspect 8:13",12,"[(aspect, 8:13)]",8:13,[],[],[intricate]
31776,Sciene fiction fight of two female robots :: Jeremy Mann :: Nasa gibli :: artstation trending :: 2 8k rendering,15,[],None,[],[],"[artstation, 8k]"
51139,"charizard but going crazy inside a liminal space, schizo mode, dmt, acid visuals, cia agent arresting you for posting on 4chan, terrorcult, dmt visuals, hor...",47,[],None,[],[],[]
94910,Portrait of Young boy with white hair by harumi hironaka and charlie bowater and edmund blair leighton,17,[],None,[],"[harumi hironaka, charlie bowater, edmund blair leighton]",[]
173858,boromir portrait :: friendly expression :: riosgraph --ar 4:5 --test,5,"[(aspect, 4:5), (test, )]",4:5,[],[],[]
115658,climbing the mountain to the volcano hyperdetailed Steven Belledin --test,9,"[(test, )]",None,[],[],[hyperdetailed]
57096,"Woman with huge glasses in advertisement for The IBM Series1 computer system, 1976",12,[],None,[],[],[]
75381,picture of mobius strip depicting opposing views of reality,9,[],None,[],[],[]
239212,Smooth surface fine-china porcelain ceramic tile::8a relief landscape painting of white clouds+moom+ chaniese palace+ willow trees+on the ceramic tile ::10v...,45,"[(aspect, 3:5)]",3:5,[],[],"[octane render, intricate, volumetric lighting]"
60844,"nighttime venice canals bridges, highly detailed, revelers, craig mullins --wallpaper",9,"[(wallpaper, )]",None,[],[],[highly detailed]


## 5. Where the two artist rules disagree

The strict and lenient rules give 14.4% vs 24.3%. These are prompts the
lenient rule flags and the strict rule does not. Some are real artists in
lower case; others are false matches such as *"by the sea"*. This is the
measurement-validity point in the Discussion.

In [8]:
strict = c["artists_strict"].map(bool)
lenient = c["artists_lenient"].map(bool)
print(f"strict only: {(strict & ~lenient).sum():,}   lenient only: {(lenient & ~strict).sum():,}   both: {(strict & lenient).sum():,}")
c.loc[lenient & ~strict, ["text", "artists_lenient"]].sample(15, random_state=SEED)

strict only: 11   lenient only: 5,751   both: 8,282


,text,artists_lenient
61093,"epic sun god, sun, sunlit, sunburn, very hot, apocalyptic, rays, angry, sunflower, mc1r, mycelium, in the style of stefan gesell and martin parr","[stefan gesell, martin parr]"
143649,"brain leaking out of someone's ear while they stare at a computer, in the style of pixar",[pixar]
739,artwork by valery slauk,[valery slauk]
41806,"deer river spirit, fantasy forest, river stream, vines, trees, flowers, fireflies, moonlight, detailed linework, blue tones, cinematic, volumetric lighting,...",[tsutomu nihei]
703,"color and light, geometric, painting by paul klee",[paul klee]
160707,"wooden tukutuku panel, in the style of sheree willman, geometric design",[sheree willman]
192916,artwork by mark gertler,[mark gertler]
27357,"Portrait of young female jisoo 김지수 キム・ジス,lisa ,jennie,rose,yoona,tzuyu, long hair, by jc leyendecker ,Trending on ArtStation and pixiv",[jc leyendecker]
79348,"girl penguin, cutesie, big eyes, surrounded by hearts, detailed",[hearts]
142832,"red samurai, chinese dragon in the sky, japanese temple surrounded by bright blue water, cloudy sky, mountains in the background, red tori gate in the foreg...",[bright blue water]


## 6. Topics: the prompts behind the top words

Refit the reference topic model (same settings as the pipeline) and show the
three prompts that load most strongly on each topic.

In [9]:
X, vocab, nmf = P.fit_topics(c)
W = nmf.transform(X)
texts = c["text"].to_numpy()
for k, row in enumerate(nmf.components_):
    words = ", ".join(vocab[np.argsort(row)[::-1][:8]])
    print(f"--- topic {k + 1}: {words}")
    for i in np.argsort(W[:, k])[::-1][:3]:
        print("   ", texts[i][:150])

--- topic 1: render, octane, unreal, engine, high, artstation, alien, photorealistic
    a Metroid made of jello, octane render —aspect 16:9
    tic-tac-toe, 3D, octane render
    brainstorm configuration, octane render
--- topic 2: style, ghibli, anime, poster, design, city, pixar, studio
    elvin nymph in the style of botero
    boxer in the style of Botero
    mustard sunscreen in the style of Cezanne
--- topic 3: artwork, van, john, peter, mike, robert, david, jean
    artwork by fujii yoshitoyo
    artwork by nagato iwasaki
    artwork by guillaume sorel
--- topic 4: art, concept, fantasy, circa, pulp, magazine, epic, nouveau
    lumberjack art concept
    ent concept art --ar 2:5
    concept art of the last of us. --ar 16:9
--- topic 5: painting, oil, matte, abstract, space, beautiful, battle, hyperrealistic
    Harambe painting himself
    painting by Matthew Lyons
    Inventions of Childhood painting --ar 2:3
--- topic 6: black, white, background, red, ink, photo, paper, dark


## 7. Hand check (validity and reliability)

We each label the same 100 random corpus prompts **without looking at the
rule's answer** and without talking about them first. That gives:

- **validity:** how often each rule agrees with a human (precision and
  recall);
- **reliability:** how often we agree with each other (Cohen's κ).

The cell below writes `validity/handcheck.csv` **once**. It never overwrites
an existing file, so labels are safe. Each of us fills in our own columns
with `1` (yes) or `0` (no):

- `artist_r1`, `artist_r2`: does the prompt name an artist, studio or
  photographer whose style it asks for?
- `booster_r1`, `booster_r2`: does it use a generic quality phrase (e.g.
  "8k", "highly detailed", "trending on artstation") rather than describing
  content?

The rule columns are kept in a separate file (`handcheck_rules.csv`) so we
are not influenced by them while labelling.

In [10]:
VALID = HERE / "validity"
VALID.mkdir(exist_ok=True)
sheet_path, rules_path = VALID / "handcheck.csv", VALID / "handcheck_rules.csv"

sample = c.sample(100, random_state=SEED)
if sheet_path.exists():
    print("already exists, not overwriting:", sheet_path)
else:
    sheet = pd.DataFrame({"row_id": sample.index, "text": sample["text"].values})
    for col in ["artist_r1", "artist_r2", "booster_r1", "booster_r2", "notes"]:
        sheet[col] = ""
    sheet.to_csv(sheet_path, index=False)
    pd.DataFrame({
        "row_id": sample.index,
        "rule_artist_strict": sample["artists_strict"].map(bool).astype(int).values,
        "rule_artist_lenient": sample["artists_lenient"].map(bool).astype(int).values,
        "rule_booster": sample["boosters"].map(bool).astype(int).values,
    }).to_csv(rules_path, index=False)
    print("wrote", sheet_path, "and", rules_path)

wrote /home/elta/repos/applied-research/assignment3/analysis/validity/handcheck.csv and /home/elta/repos/applied-research/assignment3/analysis/validity/handcheck_rules.csv


### Score the hand check

Run this after both of us have labelled. Disagreements between raters are
settled by discussion and recorded as the `final` label; until then the
cell uses rater 1 as the reference and says so.

In [11]:
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score

sheet = pd.read_csv(sheet_path).merge(pd.read_csv(rules_path), on="row_id")
labelled = sheet.dropna(subset=["artist_r1", "booster_r1"])
if labelled.empty:
    print("No labels yet: fill in", sheet_path.name)
else:
    print(f"{len(labelled)} labelled prompts")
    out = []
    for concept, rules in [("artist", ["rule_artist_strict", "rule_artist_lenient"]),
                           ("booster", ["rule_booster"])]:
        r1, r2 = f"{concept}_r1", f"{concept}_r2"
        both = labelled.dropna(subset=[r2])
        kappa = cohen_kappa_score(both[r1], both[r2]) if len(both) else np.nan
        ref = (labelled[f"{concept}_final"] if f"{concept}_final" in labelled
               else labelled[r1]).astype(int)
        for rule in rules:
            out.append({
                "concept": concept, "rule": rule,
                "human_yes_%": 100 * ref.mean(),
                "rule_yes_%": 100 * labelled[rule].mean(),
                "precision": precision_score(ref, labelled[rule], zero_division=0),
                "recall": recall_score(ref, labelled[rule], zero_division=0),
                "rater_kappa": kappa, "n_both_raters": len(both),
            })
    if "artist_final" not in labelled:
        print("No *_final columns yet: rater 1 is the reference.")
    display(pd.DataFrame(out).round(2))

No labels yet: fill in handcheck.csv
